In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import ast
from collections import Counter
from nltk.stem import WordNetLemmatizer
import re



#CARGA DE DATOA
train_data = pd.read_csv("../../Data/OnlyOneEmotion/train_emotions.csv")
test_conflictive = pd.read_csv("../../Data/OnlyOneEmotion/test_emotions_conflicting.csv")
test_clean_data = pd.read_csv("../../Data/OnlyOneEmotion/test_emotions_complete.csv")

# Convertir las cadenas de listas en listas reales
all_emotions = test_conflictive['Emotion'].apply(ast.literal_eval)

# Aplanar todas las listas en una sola lista
flattened_emotions = [emotion for sublist in all_emotions for emotion in sublist]

# Contar ocurrencias de cada emoción
emotion_counts = Counter(flattened_emotions)

#Reentrenamiento del modelo con SVM después de comprobar que es el mejor modelo, (usamos el train_data original, sin división, y ambos archivos de test, uno para etiquetar y añadirlo al train y otro para predecir y evaluar su desempeño con y sin aumento de datos)

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)
    
X_train_raw = train_data['Text'].values
X_test_conflictive_raw = test_conflictive['Text'].values
X_test_clean_raw = test_clean_data['Text'].values

# Aplicar lematización a los textos
X_train_lem = [preprocess_text(text) for text in X_train_raw]
X_test_conflictive_lem = [preprocess_text(text) for text in X_test_conflictive_raw]
X_test_clean_lem = [preprocess_text(text) for text in X_test_clean_raw]

#Vectorizar los textos
vectorizer = TfidfVectorizer(max_features=10000, lowercase=True, strip_accents='unicode')

# Reentrenamiento del modelo con SVM
X_full_train = vectorizer.fit_transform(X_train_lem)
X_test_clean = vectorizer.transform(X_test_clean_lem)
X_test_conflictive = vectorizer.transform(X_test_conflictive_lem)

y_full_train = train_data['Emotion']
y_test_clean = test_clean_data['Emotion']
y_test_conflictive = test_conflictive['Emotion']

clf = SVC(kernel='linear', class_weight='balanced')

print("Reentrenando el modelo con SVM...")
clf.fit(X_full_train, y_full_train)

#Predicción sobre el conjunto test limpio
y_pred_test_clean = clf.predict(X_test_clean)
print(f"Predicciones en el conjunto test limpio:\n {classification_report(y_test_clean, y_pred_test_clean, zero_division=0)}")

print(f"\nPrecisión en test limpio (SVM): {accuracy_score(y_test_clean, y_pred_test_clean):.4f}")
dict_test_clean = classification_report(y_test_clean, y_pred_test_clean, output_dict=True, zero_division=0)
f1_macro_test_clean = dict_test_clean["macro avg"]["f1-score"]
recall_macro_test_clean = dict_test_clean["macro avg"]["recall"]
print(f"F1 Score en test limpio (macro avg): {f1_macro_test_clean:.4f}")
print(f"Recall Score en test limpio (macro avg): {recall_macro_test_clean:.4f}")




Reentrenando el modelo con SVM...
Predicciones en el conjunto test limpio:
               precision    recall  f1-score   support

           0       0.59      0.54      0.56       348
           1       0.75      0.77      0.76       186
           2       0.37      0.44      0.40       131
           3       0.21      0.20      0.20       194
           4       0.24      0.28      0.26       236
           5       0.18      0.38      0.25        86
           6       0.21      0.36      0.26        97
           7       0.24      0.38      0.29       176
           8       0.30      0.46      0.36        56
           9       0.15      0.23      0.18        88
          10       0.26      0.42      0.32       195
          11       0.42      0.49      0.45        76
          12       0.23      0.26      0.24        23
          13       0.24      0.53      0.33        57
          14       0.60      0.65      0.62        65
          15       0.93      0.87      0.90       260
     

In [12]:


# Predicción sobre el conjunto test conflictivo
y_pred_test_conflictive = clf.predict(X_test_conflictive)

# Crear DataFrame con las predicciones del test conflictivo
new_test_conflictive_data = pd.DataFrame({
    'Text': test_conflictive['Text'],
    'Emotion': y_pred_test_conflictive
})

#Crear otro DatraFrame para inspeccion manual de las predicciones del test conflictivo
comparation_df = pd.DataFrame({
    'Text': test_conflictive['Text'],
    'Original Emotion': test_conflictive['Emotion'],
    'Predicted Emotion': y_pred_test_conflictive
})

print(f"Predicciones del test conflictivo:\n{new_test_conflictive_data.head()}")

print("\nCuenta de emociones en la predicción:\n")
print(comparation_df["Predicted Emotion"].value_counts())

# Guardar las predicciones del test conflictivo en un archivo CSV
comparation_df.to_csv("../../Data/OnlyOneEmotion/ToCompareResults.csv", index=False)

#Concatenar el DataFrame de test conflictivo con el train original para volver a entrenar el modelo
new_train_data_conflictive = pd.concat([train_data, new_test_conflictive_data], ignore_index=True)


# Asegurar que la columna Original Emotion esté en formato lista
comparation_df['Original Emotion'] = comparation_df['Original Emotion'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Filtrar solo los casos donde la predicción esté contenida en la lista de emociones originales
correct_predictions_df = comparation_df[
    comparation_df.apply(lambda row: row['Predicted Emotion'] in row['Original Emotion'], axis=1)
].copy()

print("\nFilas donde la predicción coincide con al menos una emoción original:")
print(correct_predictions_df.head())

print(f"\nNúmero total de aciertos parciales: {len(correct_predictions_df)} / {len(comparation_df)}")


correct_predictions_df.to_csv("../../Data/OnlyOneEmotion/PrediccionesCorrectas.csv", index=False)

#Class weight balanced --> No funcionó

#Hacer una limpieza del etiquetado en bruto de aquellos que si predijo de forma correcta.

Predicciones del test conflictivo:
                                                Text  Emotion
0  We need more boards and to create a bit more s...       20
1  Aww... she'll probably come around eventually,...        1
2  Shit, I guess I accidentally bought a Pay-Per-...       27
3  Maybe that’s what happened to the great white ...        0
4  I never thought it was at the same moment, but...       22

Cuenta de emociones en la predicción:

Predicted Emotion
15    707
27    663
18    601
0     545
1     503
7     420
4     402
10    396
3     333
2     321
20    311
5     307
6     303
17    293
26    225
9     219
25    208
22    208
13    198
24    189
11    182
8     176
14    117
12     42
21     24
19     21
16     15
23     10
Name: count, dtype: int64

Filas donde la predicción coincide con al menos una emoción original:
                                                Text Original Emotion  \
0  We need more boards and to create a bit more s...          [8, 20]   
1  Aww... sh

In [13]:
#Segundo reentrenamiento del modelo con SVM usando el train con prediccions limpias
print(correct_predictions_df.head())


print("Valores por emoción:")
print(correct_predictions_df['Predicted Emotion'].value_counts())

correct_predictions_df['Emotion'] = correct_predictions_df['Predicted Emotion']

toConcat_df = pd.DataFrame({
    'Text': correct_predictions_df['Text'],
    'Emotion': correct_predictions_df['Emotion'],
})


print(toConcat_df.head())

new_train_data_correct = pd.concat([train_data, toConcat_df], ignore_index=True)

new_train_data_correct['Text'] = new_train_data_correct['Text'].apply(preprocess_text)

X_new_train_correct = vectorizer.fit_transform(new_train_data_correct['Text'])
y_new_train_correct = new_train_data_correct['Emotion']

clf = SVC(kernel='linear', class_weight='balanced')

print("Reentrenando el modelo con SVM con el nuevo conjunto de entrenamiento (limpio)...")
clf.fit(X_new_train_correct, y_new_train_correct)

# Predicción sobre el conjunto test limpio con el nuevo modelo
y_pred_new_test_clean_correct = clf.predict(X_test_clean)
print(f"Predicciones en el conjunto test limpio con el nuevo modelo:\n {classification_report(y_test_clean, y_pred_new_test_clean_correct, zero_division=0)}")

print(f"\nPrecisión en test limpio con aumento correcto (SVM): {accuracy_score(y_test_clean, y_pred_new_test_clean_correct):.4f}")
dict_test_clean_correct = classification_report(y_test_clean, y_pred_new_test_clean_correct, output_dict=True, zero_division=0)
f1_macro_test_clean_correct = dict_test_clean_correct["macro avg"]["f1-score"]
recall_macro_test_clean_correct = dict_test_clean_correct["macro avg"]["recall"]
print(f"F1 Score en test limpio con aumento correcto (macro avg): {f1_macro_test_clean_correct:.4f}")
print(f"Recall Score en test limpio con aumento correcto (macro avg): {recall_macro_test_clean_correct:.4f}")




                                                Text Original Emotion  \
0  We need more boards and to create a bit more s...          [8, 20]   
1  Aww... she'll probably come around eventually,...           [1, 4]   
5                            I miss them being alive         [16, 25]   
6        Ok, then what the actual fuck is your plan?           [2, 7]   
7                    aw, thanks! I appreciate that!           [0, 15]   

   Predicted Emotion  
0                 20  
1                  1  
5                 25  
6                  2  
7                 15  
Valores por emoción:
Predicted Emotion
15    686
18    491
1     440
0     423
27    298
20    223
17    209
7     208
4     204
2     198
3     192
10    163
25    145
26    145
5     132
6     129
24    123
11     93
8      89
13     89
9      88
22     85
14     67
12     27
21     18
19     12
16      9
23      2
Name: count, dtype: int64
                                                Text  Emotion
0  We need more 

In [14]:
#Tercer reentrenamiento del modelo con SVM usando el nuevo train que incluye las predicciones del test conflictivo
new_train_data_conflictive['Text'] = new_train_data_conflictive['Text'].apply(preprocess_text)

X_new_train_conflictive = vectorizer.fit_transform(new_train_data_conflictive['Text'])
y_new_train_confilctive = new_train_data_conflictive['Emotion']

clf = SVC(kernel='linear', class_weight='balanced')

print("Reentrenando el modelo con SVM con el nuevo conjunto de entrenamiento (conflictivo)...")
clf.fit(X_new_train_conflictive, y_new_train_confilctive)

# Predicción sobre el conjunto test limpio con el nuevo modelo
y_pred_new_test_clean_conflictive = clf.predict(X_test_clean)
print(f"Predicciones en el conjunto test conflictivo con el nuevo modelo: {classification_report(y_test_clean, y_pred_new_test_clean_conflictive, zero_division=0)}")
print(f"\nPrecisión en test limpio con aumento conflictivo (SVM): {accuracy_score(y_test_clean, y_pred_new_test_clean_conflictive):.4f}")
dict_test_clean_conflictive = classification_report(y_test_clean, y_pred_new_test_clean_conflictive, output_dict=True, zero_division=0)
f1_macro_test_clean_conflictive = dict_test_clean_conflictive["macro avg"]["f1-score"]
recall_macro_test_clean_conflictive = dict_test_clean_conflictive["macro avg"]["recall"]
print(f"F1 Score en test limpio con aumento conflictivo (macro avg): {f1_macro_test_clean_conflictive:.4f}")
print(f"Recall Score en test limpio con aumento conflictivo (macro avg): {recall_macro_test_clean_conflictive:.4f}")

Reentrenando el modelo con SVM con el nuevo conjunto de entrenamiento (conflictivo)...
Predicciones en el conjunto test conflictivo con el nuevo modelo:               precision    recall  f1-score   support

           0       0.04      0.01      0.01       348
           1       0.00      0.00      0.00       186
           2       0.06      0.02      0.02       131
           3       0.00      0.00      0.00       194
           4       0.05      0.01      0.01       236
           5       0.09      0.01      0.02        86
           6       0.08      0.01      0.02        97
           7       0.05      0.01      0.01       176
           8       0.17      0.04      0.06        56
           9       0.09      0.02      0.04        88
          10       0.10      0.01      0.01       195
          11       0.01      0.01      0.01        76
          12       0.00      0.00      0.00        23
          13       0.00      0.00      0.00        57
          14       0.05      0.03   

In [15]:

# --- DIRECTORIO DE SALIDA DE LOS PLOTS ---
output_dir = "../../Plots/Experiment2/"


import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Función para graficar matriz de confusión
def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))

    plt.figure(figsize=(12, 10))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues", 
        xticklabels=labels,
        yticklabels=labels,
        cbar=True,
        annot_kws={"size": 6}
    )
    ax.set_title(f'Matriz de Confusión - {title}', fontsize=12)
    ax.set_xlabel('Etiqueta Predicha', fontsize=10)
    ax.set_ylabel('Etiqueta Verdadera', fontsize=10)

    # Reducir tamaño de etiquetas en los ejes
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=6)
    
    plt.tight_layout()
    # Guardar archivo
    filename = title.replace(" ", "_").replace("(", "").replace(")", "").lower() + ".png"
    plt.savefig(output_dir + filename)
    plt.close()



emotion_labels = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity',
    'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]
#Sin aumento de datos
plot_confusion_matrix(
    y_true=y_test_clean,
    y_pred=y_pred_test_clean,
    labels=emotion_labels,
    title="Original sin aumento de datos"
)

#Con aumento limpio
plot_confusion_matrix(
    y_true=y_test_clean,
    y_pred=y_pred_new_test_clean_correct,
    labels=emotion_labels,
    title="Con aumento de datos (predicciones correctas)"
)

#Con aumento conflictivo
plot_confusion_matrix(
    y_true=y_test_clean,
    y_pred=y_pred_new_test_clean_conflictive,
    labels=emotion_labels,
    title="Con aumento de datos (predicciones conflictivas)"
)



    
    